In [ ]:
# Celda 1: Importaciones y Configuración
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

sys.path.append(os.path.abspath('..'))

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Celda 2: Cargar modelo y datos
model = joblib.load('models/best_model.pkl')
preprocessor = joblib.load('models/preprocessor.pkl')

df = pd.read_csv('data/raw/train.csv')
from src.features import feature_engineering
df = feature_engineering(df)

df['SalePrice'] = np.log1p(df['SalePrice'])
y = df['SalePrice']
X = df.drop(['SalePrice', 'Id'], axis=1, errors='ignore')

X_processed = preprocessor.transform(X)

print("Modelo y datos cargados correctamente")

In [ ]:
# Celda 3: Predicciones y Residuales
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42
)

y_pred = model.predict(X_test)

# Volver a escala original (dólares)
y_test_orig = np.expm1(y_test)
y_pred_orig = np.expm1(y_pred)

residuals = y_test_orig - y_pred_orig

In [ ]:
# Celda 4: Métricas Detalladas
from sklearn.metrics import mean_squared_log_error, mean_absolute_error, r2_score

rmsle = np.sqrt(mean_squared_log_error(y_test_orig, y_pred_orig))
mae = mean_absolute_error(y_test_orig, y_pred_orig)
r2 = r2_score(y_test, y_pred)

print("="*60)
print("📊 MÉTRICAS FINALES")
print("="*60)
print(f"RMSLE  : {rmsle:.5f}")
print(f"MAE    : ${mae:,.2f} USD")
print(f"R²     : {r2:.4f}")
print("="*60)

In [ ]:
# Celda 5: Gráfico de Residuales
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Residuals vs Predicted
axes[0].scatter(y_pred_orig, residuals, alpha=0.6)
axes[0].axhline(y=0, color='r', linestyle='--')
axes[0].set_xlabel('Precio Predicho')
axes[0].set_ylabel('Residual (Real - Predicho)')
axes[0].set_title('Residuals vs Predicted')

# Distribución de Residuales
sns.histplot(residuals, kde=True, ax=axes[1])
axes[1].set_title('Distribución de Residuales')
plt.show()

In [ ]:
# Celda 6: Ejemplos de Predicciones (Tabla comparativa)
results = pd.DataFrame({
    'Actual Price': y_test_orig,
    'Predicted Price': y_pred_orig,
    'Absolute Error': np.abs(residuals),
    'Percentage Error': np.abs(residuals) / y_test_orig * 100
})

print("🔍 Ejemplos de Predicciones:")
display(results.head(10).round(2))

print(f"\nError promedio: ${results['Absolute Error'].mean():,.2f}")
print(f"Error porcentual promedio: {results['Percentage Error'].mean():.2f}%")

In [ ]:
# Celda 7: Peores predicciones (Outliers)
worst_predictions = results.sort_values('Absolute Error', ascending=False).head(5)
print("❌ Peores 5 predicciones:")
display(worst_predictions.round(2))

In [ ]:
# Celda 8: Guardar resultados
results.to_csv('reports/predictions_analysis.csv', index=False)
print("📁 Resultados guardados en reports/predictions_analysis.csv")